# EEG-to-Image: End-to-End Reproducibility Notebook

This notebook reproduces the headline numbers in the report:
- 200-way zero-shot retrieval Top-1 / Top-5
- Image reconstruction (SSIM, CLIP, ...) over 10 seeds

All heavy work (CLIP feature caching, training, image generation) is run via `slurm_scripts/`.
This notebook only loads the saved checkpoints + outputs and prints the metrics.

## 1. Setup

In [ ]:
import sys, os, json
from pathlib import Path

PROJECT_ROOT = Path('/hpc2hdd/home/ckwong627/workdir/Class/DSAA2012-Deep_Learning/ChiKitWONG/Assignments/Project/DL_Project')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'codes'))

import torch
import matplotlib.pyplot as plt
from config import DEFAULT_CONFIG
from model import UnifiedModel, count_parameters
from utils import set_seed, compute_retrieval_metrics, eval_images, summarize_metrics_over_seeds, load_real_test_images

cfg = DEFAULT_CONFIG
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(0)
print('device:', device)

## 2. Load trained encoder and inspect

In [ ]:
model = UnifiedModel(cfg, alpha=1.0, beta=1.0, learnable_loss_weights=False).to(device)
ckpt_path = cfg.ckpt_dir / 'phase2_main_best.pt'
state = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(state['model'], strict=False)
model.eval()
print(f'loaded {ckpt_path}')
print(f'#params: {count_parameters(model):,}')

## 3. Retrieval evaluation (200-way zero-shot)

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from data import EEGImageDataset, collate_eeg_batch, load_eeg_dataset

test_ds = load_eeg_dataset(
    data_directory=cfg.data_dir,
    split='test',
    avg_trials=cfg.avg_trials,
    eeg_channel_jsonl=cfg.eeg_channels_jsonl,
)
test_features = torch.load(cfg.cache_dir / 'clip_test_features.pt', weights_only=False)
test_set = EEGImageDataset(test_ds, test_features, augmentation=None)
loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=0, collate_fn=collate_eeg_batch)

eeg_embs, clip_embs = [], []
with torch.no_grad():
    for batch in loader:
        eeg = batch['eeg'].to(device)
        clip = batch['clip'].to(device)
        eeg_embs.append(F.normalize(model.encode(eeg), dim=-1).cpu())
        clip_embs.append(F.normalize(clip, dim=-1).cpu())
eeg_all = torch.cat(eeg_embs)
clip_all = torch.cat(clip_embs)
logits = eeg_all @ clip_all.T
metrics = compute_retrieval_metrics(logits)
print(f"Top-1: {metrics['top1_acc']:.4f}")
print(f"Top-5: {metrics['top5_acc']:.4f}")

## 4. Reconstruction evaluation (10 seeds, official metrics)

In [ ]:
recon_path = cfg.output_dir / 'recon_images_phase2_main.pt'
bundle = torch.load(recon_path, weights_only=False)
images = bundle['images']  # [n_seeds, 200, 3, 256, 256]
image_ids = bundle['image_ids']
seeds = bundle['seeds']
print('images shape:', images.shape, '  seeds:', seeds)

real = load_real_test_images(image_ids, cfg.test_image_dir, size=cfg.recon_eval_size)
print('real images shape:', real.shape)

In [ ]:
per_seed = []
for s_idx, seed in enumerate(seeds):
    print(f'--- seed {seed} ---')
    m = eval_images(real_images=real, fake_images=images[s_idx], device=device)
    m['seed'] = int(seed)
    per_seed.append(m)
    print(m)

summary = summarize_metrics_over_seeds(per_seed)
print('\n=== reconstruction summary ===')
for k, v in summary.items():
    print(f"{k}: {v['mean']:.4f} ± {v['std']:.4f}")

## 5. Qualitative grid (8 success cases)

In [ ]:
n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(2*n_show, 5))
for j in range(n_show):
    axes[0, j].imshow(real[j].permute(1, 2, 0).numpy())
    axes[0, j].set_title(image_ids[j], fontsize=7)
    axes[0, j].axis('off')
    axes[1, j].imshow(images[0, j].permute(1, 2, 0).numpy())
    axes[1, j].axis('off')
axes[0, 0].set_ylabel('real', fontsize=10)
axes[1, 0].set_ylabel('recon', fontsize=10)
plt.tight_layout()
plt.show()